General information

In [ ]:
# Time/year/month etc. use time_period as a parameter
# everything else, use filters as a parameter

Import libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import time
import os

Import class

In [2]:
import sys
sys.path.insert(0, '..')

from src.fetcher import SSBcsvFetcher

fetcher = SSBcsvFetcher()

Print information of table

In [ ]:
# Print nicely formatted info
fetcher.print_table_info('12292')

Inspect filters

In [ ]:
# Get dimension details for Region
dimensions = fetcher.get_dimensions('12292')
region_values = dimensions['KOKkommuneregion0000']['values']
statistics_variable_values = dimensions['ContentsCode']['values']

# Display regions as DataFrame
region_df = pd.DataFrame(list(region_values.items()), columns=['Code', 'Name'])
print("Regions:")
print(region_df.to_string())

# Display statistics variables as DataFrame
print("\n\nStatistics Variables:")
stats_df = pd.DataFrame(list(statistics_variable_values.items()), columns=['Code', 'Name'])
print(stats_df.to_string())

Klass check

In [3]:
# Fetch counties from KLASS API
counties_df = fetcher.get_counties()
print("Counties from KLASS API 2026:")
print(counties_df.to_string())

# Get just the codes for filtering
county_codes = fetcher.get_county_codes()
print(f"\nCounty codes: {county_codes}")

# Get code -> name mapping
county_mapping = fetcher.get_county_mapping()
print(f"\nCounty mapping: {county_mapping}")

INFO:src.klass:Fetching classification 104 from KLASS API
ERROR:src.klass:Error fetching classification 104: 502 Server Error: Bad Gateway for url: https://data.ssb.no/api/klass/v1/classifications/104/codes?language=no&format=json


HTTPError: 502 Server Error: Bad Gateway for url: https://data.ssb.no/api/klass/v1/classifications/104/codes?language=no&format=json

Fetch DataFrame

In [ ]:
# Fetch data using region codes from KLASS API (authoritative 2026 data)
# Get all county codes
county_codes = fetcher.get_county_codes()
county_mapping = fetcher.get_county_mapping()

print(f"Found {len(county_codes)} counties from KLASS API:")
print(county_mapping)

# Get all statistics variable codes
dimensions = fetcher.get_dimensions('12292')
statistics_variable_values = dimensions['ContentsCode']['values']

# Fetch only these two specific variables
stats_to_fetch = [
    ('KOSkjernetotalt0000', 'Hjemmetjenester - brukere i alt (antall)'),
    ('KOShjelptkjernet0000', 'Hjemmetjenestebrukere, omfattende bistandsbehov, i alt (antall)')
]

# Create a dictionary to store dataframes with clean names
dfs = {}
failed_variables = []

# Fetch data for each statistics variable using county codes from KLASS
for code, name in stats_to_fetch:
    try:
        df = fetcher.fetch_clean_table(
            '12292',
            time_period='*',
            filters={
                'KOKkommuneregion0000': ','.join(county_codes),
                'ContentsCode': code
            }
        )
        
        # Clean the dataframe name
        clean_name = name.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_').replace(',', '')
        clean_name = clean_name[:60]
        
        dfs[clean_name] = df
        print(f"✓ Loaded: {clean_name}")
        
        # Add delay to avoid rate limiting
        time.sleep(10)
        
    except Exception as e:
        print(f"✗ Failed to load {code}: {str(e)[:80]}")
        failed_variables.append(code)

print(f"\nTotal dataframes loaded: {len(dfs)}")
print(f"Failed variables: {len(failed_variables)}")

Store dfs

In [ ]:
# Create output directory if it doesn't exist
output_dir = r'C:\Users\matia\Desktop\ssb\eda\output'
os.makedirs(output_dir, exist_ok=True)

# Function to sanitize filename
def sanitize_filename(name):
    # Remove/replace problematic characters
    name = name.replace('/', '_').replace('\\', '_')  # Path separators
    name = name.replace(':', '_')  # Drive letters
    name = name.replace('*', '_').replace('?', '_')   # Wildcards
    name = name.replace('"', '_').replace("'", '_')   # Quotes
    name = name.replace('<', '_').replace('>', '_')   # Angle brackets
    name = name.replace('|', '_')  # Pipe
    return name.strip()

# Save each dataframe to CSV
for name, df in dfs.items():
    safe_name = sanitize_filename(name)
    filename = f"{output_dir}/{safe_name}.csv"
    df.to_csv(filename, index=False, sep=';', encoding='iso-8859-1')
    print(f"✓ Saved: {safe_name}.csv")

print(f"\nAll {len(dfs)} dataframes saved to '{output_dir}/'")

# Also create a summary file listing all dataframes
summary_df = pd.DataFrame({
    'DataFrame_Name': list(dfs.keys()),
    'Safe_Filename': [sanitize_filename(name) for name in dfs.keys()],
    'Rows': [len(df) for df in dfs.values()],
    'Columns': [len(df.columns) for df in dfs.values()]
})

summary_df.to_csv(f'{output_dir}/SUMMARY.csv', index=False, sep=';', encoding='iso-8859-1')
print(f"✓ Created summary: {output_dir}/SUMMARY.csv")

Read a Dataframe

In [ ]:
df_test = pd.read_csv(r'C:\Users\matia\Desktop\ssb\eda\output\Hjemmetjenester___brukere_i_alt_antall.csv')

Simple cleaning

In [ ]:
# drop first row in df
df = df.iloc[1:]

# drop first column in df
df = df.iloc[:, 1:]

Create grouped horizontal barchart (age y axis) and their count with years 2020 to 2025 in x axis

In [ ]:
# Filter columns for years 2020-2025
year_columns = [col for col in df.columns if str(col).isdigit() and 2020 <= int(col) <= 2025]
df_filtered = df.set_index('alder')[year_columns].astype(float)

# Define colors
colors = ['#e61414', '#E7A19C', '#99daf0', '#d6ecc0', '#D6D6D6', '#50b8e7']  # Extra color at end

# Create grouped horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 8))

df_filtered.plot(kind='barh', ax=ax, width=0.8, color=colors)

ax.set_xlabel('Befolkningstall', fontsize=12)
ax.set_ylabel('Aldersgrupper', fontsize=12)
ax.set_title('Befolkningstelling i Norge (2020-2025)', fontsize=14)
ax.legend(title='År', bbox_to_anchor=(1.05, 1), loc='upper left')

# Format x-axis to show millions
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.2f}M'))

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt=lambda x: f'{x/1e6:.2f}M', padding=3)

plt.tight_layout()
plt.show()